# Financial News Sentiment Analysis

**Author:** Elif Naz Turan  
**Model:** ProsusAI/FinBERT  
**Dataset:** Telco news from Alpha Vantage / yfinance / sample dataset

---

## Outline
1. [Setup & Data Loading](#1)
2. [Sentiment Scoring with FinBERT](#2)
3. [Sentiment Distribution Analysis](#3)
4. [Daily & Weekly Trend Analysis](#4)
5. [Stock Price vs. Sentiment Overlay](#5)
6. [Correlation Analysis](#6)
7. [Key Findings](#7)

## 1 — Setup & Data Loading <a id='1'></a>

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold'})

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.data_loader     import load_news
from src.sentiment_model import score_dataframe, LABEL_COLORS
from src.stock_data      import load_prices, price_summary
from src.analysis        import aggregate_sentiment, merge_sentiment_prices, compute_correlation
from src.utils           import save_scored, load_scored

TICKER = 'AAPL'
DAYS   = 180

# Load or fetch+score
df_scored = load_scored(TICKER)
if df_scored is None:
    print('Fetching and scoring news …')
    news      = load_news(TICKER, days=DAYS)
    df_scored = score_dataframe(news)
    save_scored(df_scored, TICKER)
else:
    print(f'Loaded cached results ({len(df_scored)} headlines)')

df_scored['date'] = pd.to_datetime(df_scored['date'])
print(f'Shape: {df_scored.shape}')
df_scored.head()

## 2 — Sentiment Scoring with FinBERT <a id='2'></a>

In [ ]:
# Score distribution overview
print('Sentiment label distribution:')
print(df_scored['sentiment_label'].value_counts())
print(f"\nMean sentiment score : {df_scored['sentiment_score'].mean():.4f}")
print(f"Median sentiment score: {df_scored['sentiment_score'].median():.4f}")
print(f"Std  sentiment score : {df_scored['sentiment_score'].std():.4f}")
df_scored[['sentiment_score','prob_positive','prob_negative','prob_neutral','confidence']].describe().round(4)

In [ ]:
# Top 5 most positive and most negative headlines
print('Top 5 most POSITIVE headlines:')
for _, row in df_scored.nlargest(5, 'sentiment_score').iterrows():
    print(f"  [{row['sentiment_score']:+.3f}] {row['headline']}")

print('\nTop 5 most NEGATIVE headlines:')
for _, row in df_scored.nsmallest(5, 'sentiment_score').iterrows():
    print(f"  [{row['sentiment_score']:+.3f}] {row['headline']}")

## 3 — Sentiment Distribution Analysis <a id='3'></a>

In [ ]:
palette = {'positive': '#06d6a0', 'negative': '#ef233c', 'neutral': '#8ecae6'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'{TICKER} — FinBERT Sentiment Overview', fontsize=14)

# Bar: label counts
counts = df_scored['sentiment_label'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=[palette[l] for l in counts.index], edgecolor='white')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 0.5, str(val), ha='center', fontsize=11)
axes[0].set_title('Headline Count by Sentiment')
axes[0].set_ylabel('Count')

# Hist: score distribution
for label, grp in df_scored.groupby('sentiment_label'):
    axes[1].hist(grp['sentiment_score'], bins=25, alpha=0.6,
                 color=palette[label], label=label, edgecolor='none')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Sentiment Score Distribution')
axes[1].set_xlabel('Score (P(pos) − P(neg))')
axes[1].legend()

# Scatter: P(positive) vs P(negative)
for label, grp in df_scored.groupby('sentiment_label'):
    axes[2].scatter(grp['prob_positive'], grp['prob_negative'],
                    c=palette[label], alpha=0.5, s=20, label=label, edgecolors='none')
axes[2].plot([0,1],[0,1],'k--',alpha=0.3)
axes[2].set_xlabel('P(Positive)')
axes[2].set_ylabel('P(Negative)')
axes[2].set_title('Probability Space')
axes[2].legend(markerscale=2)

plt.tight_layout()
os.makedirs('../outputs/charts', exist_ok=True)
plt.savefig(f'../outputs/charts/{TICKER}_sentiment_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confidence by label
fig, ax = plt.subplots(figsize=(9, 4))
for label, grp in df_scored.groupby('sentiment_label'):
    sns.kdeplot(grp['confidence'], ax=ax, fill=True, alpha=0.4,
                color=palette[label], label=label)
ax.set_title('FinBERT Confidence Score by Sentiment Class')
ax.set_xlabel('Confidence (max probability)')
ax.legend(title='Sentiment')
plt.tight_layout()
plt.show()

## 4 — Daily & Weekly Trend Analysis <a id='4'></a>

In [ ]:
df_daily  = aggregate_sentiment(df_scored, freq='D')
df_weekly = aggregate_sentiment(df_scored, freq='W')

print(f'Daily periods:  {len(df_daily)}')
print(f'Weekly periods: {len(df_weekly)}')
df_daily.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)
fig.suptitle(f'{TICKER} — Sentiment Trends', fontsize=14)

# Daily with MA
ax = axes[0]
ax.fill_between(df_daily['date'],
                df_daily['mean_score'] - df_daily['std_score'].fillna(0),
                df_daily['mean_score'] + df_daily['std_score'].fillna(0),
                alpha=0.15, color='#3a86ff', label='±1 SD')
ax.plot(df_daily['date'], df_daily['mean_score'], alpha=0.5, color='#3a86ff', linewidth=1)
ax.plot(df_daily['date'], df_daily['score_ma7'], color='#ff006e', linewidth=2, label='7-day MA')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Daily Sentiment Score')
ax.set_ylabel('Score')
ax.legend()
ax.grid(True, alpha=0.3)

# Stacked bar weekly
ax = axes[1]
ax.bar(df_weekly['date'], df_weekly['pct_positive'], color='#06d6a0', label='Positive', width=5)
ax.bar(df_weekly['date'], df_weekly['pct_neutral'],  color='#8ecae6', label='Neutral',
       bottom=df_weekly['pct_positive'], width=5)
ax.bar(df_weekly['date'], df_weekly['pct_negative'], color='#ef233c', label='Negative',
       bottom=df_weekly['pct_positive'] + df_weekly['pct_neutral'], width=5)
ax.set_title('Weekly Sentiment Composition (%)')
ax.set_ylabel('Percentage')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'../outputs/charts/{TICKER}_sentiment_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 — Stock Price vs. Sentiment Overlay <a id='5'></a>

In [ ]:
df_prices = load_prices(TICKER, days=DAYS)
print(f'Price data: {len(df_prices)} trading days')
print(price_summary(df_prices))

df_merged = merge_sentiment_prices(df_daily, df_prices)
print(f'Merged rows: {len(df_merged)}')
df_merged.head()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.plot(df_merged['Date'], df_merged['price_norm'], color='#3a86ff', linewidth=2, label='Price (base=100)')
ax2.plot(df_merged['Date'], df_merged['score_ma7'], color='#ff006e', linewidth=2, linestyle='--', label='Sentiment MA-7')
ax2.axhline(0, color='gray', linestyle=':', alpha=0.5)

ax1.set_xlabel('Date')
ax1.set_ylabel('Normalised Price (base 100)', color='#3a86ff')
ax2.set_ylabel('Sentiment Score', color='#ff006e')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title(f'{TICKER} — Price vs. Sentiment MA-7', fontsize=13)
plt.tight_layout()
plt.savefig(f'../outputs/charts/{TICKER}_price_vs_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Correlation Analysis <a id='6'></a>

In [ ]:
from scipy import stats

corr = compute_correlation(df_merged)
print('Sentiment Score → Next-Day Return Correlation')
print('=' * 50)
for k, v in corr.items():
    print(f'  {k:<22}: {v}')

print('\n⚠️  Note: Correlation does not imply causation.')
print('   These results are exploratory only and should not drive trading decisions.')

In [ ]:
clean = df_merged[['mean_score','return_next']].dropna()
x = clean['mean_score']
y = clean['return_next'] * 100

slope, intercept, r, p, se = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x, y, alpha=0.5, color='#3a86ff', s=30, edgecolors='none', label='Observations')
ax.plot(x_line, slope * x_line + intercept, color='#ff006e', linewidth=2, label=f'OLS fit (r={r:.3f})')
ax.axhline(0, color='gray', linestyle='--', alpha=0.4)
ax.axvline(0, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('Daily Mean Sentiment Score')
ax.set_ylabel('Next-Day Return (%)')
ax.set_title(f'{TICKER} — Sentiment vs. Next-Day Return (Pearson r={r:.3f}, p={p:.4f})')
ax.legend()
plt.tight_layout()
plt.savefig(f'../outputs/charts/{TICKER}_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 — Key Findings <a id='7'></a>

In [ ]:
n     = len(df_scored)
n_pos = (df_scored['sentiment_label'] == 'positive').sum()
n_neg = (df_scored['sentiment_label'] == 'negative').sum()
n_neu = (df_scored['sentiment_label'] == 'neutral').sum()
mean_score = df_scored['sentiment_score'].mean()

print('=' * 65)
print(f'  SENTIMENT ANALYSIS FINDINGS — {TICKER}')
print('=' * 65)
print(f'\n  Headlines scored : {n}')
print(f'  Positive         : {n_pos} ({n_pos/n:.1%})')
print(f'  Negative         : {n_neg} ({n_neg/n:.1%})')
print(f'  Neutral          : {n_neu} ({n_neu/n:.1%})')
print(f'  Mean score       : {mean_score:+.4f}')
print(f'  Overall tone     : {"Bullish" if mean_score > 0.05 else ("Bearish" if mean_score < -0.05 else "Neutral")}')

if corr.get('pearson_r'):
    sig = "(statistically significant)" if corr['pearson_sig'] else "(not significant)"
    print(f'\n  Pearson r (sentiment → next return): {corr["pearson_r"]:+.4f} {sig}')
    print(f'  Spearman r                          : {corr["spearman_r"]:+.4f}')

print(f'\n  ⚠️  Reminder: do not trade on this analysis.')
print('=' * 65)